# Train LoRA SFT — Qwen3.5-0.8B Cloud Classifier (Resumable, configurable epochs)

This notebook trains a LoRA adapter on `Qwen3.5-0.8B` for a configurable number of
epochs of single-label cloud classification SFT. Set `TOTAL_EPOCHS` and
`MILESTONE_EPOCHS` in the Configuration cell to control the run length.

| Feature | This notebook |
|---------|--------------|
| LR schedule | One continuous cosine decay across **all epochs** |
| Optimizer state | Saved and restored on resume — Adam m/v never reset |
| Scheduler state | Saved and restored on resume — cosine picks up exactly where it left off |
| RNG state | Saved and restored — same data ordering as a continuous run |
| Milestone adapters | LoRA adapter exported at each milestone epoch to a standalone folder |
| Session resume | Run this notebook again after a crash — it auto-detects the last checkpoint |
| Extend training | Increase `TOTAL_EPOCHS` and re-run to continue past the last checkpoint |
| Milestone weight provenance | `load_best_model_at_end=False` — milestone adapters are the **actual** epoch N weights |

**Directory layout (all under `/kaggle/working/qwen35-cloud-lora/`):**
```
trainer_checkpoints/      ← Trainer saves here each epoch (optimizer + scheduler + rng)
  checkpoint-NNN/         ← rotating checkpoints (keep last 3)
  ...
milestone_adapters/       ← Never deleted; one clean adapter per milestone epoch
  adapter_epochXX/
  ...
```


## 1. Install Dependencies

In [ ]:
"""
# Run this cell ONCE. Restart the runtime when it finishes, then continue from cell 2.
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", *args])

subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y",
                 "verl", "vllm", "mistral-common", "trl",
                 "peft", "transformers", "tokenizers", "accelerate", "bitsandbytes"])

pip("install", "-U", "--no-cache-dir", "pip<25.1")

# transformers from GitHub dev (required for Qwen3.5 model_type="qwen3_5").
pip("install", "-U", "--no-cache-dir", "git+https://github.com/huggingface/transformers.git")

# peft from GitHub dev — stable PyPI peft fails with dev transformers.
pip("install", "-U", "--no-cache-dir",
    "git+https://github.com/huggingface/peft.git",
    "accelerate>=1.0.0", "datasets>=3.0.0", "safetensors>=0.4.0",
    "scikit-learn>=1.6.0", "seaborn>=0.13.0", "matplotlib>=3.8.0,<3.11.0",
    "pillow>=10.4.0,<12.0.0", "torchvision", "qwen-vl-utils>=0.0.10")

pip("install", "--force-reinstall", "--no-cache-dir",
    "numpy==2.1.3", "scipy==1.14.1", "scikit-learn==1.7.2", "pillow>=10.4.0,<12.0.0")

subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "torchao", "torchaudio"])

print("Done. Restart the runtime, then continue from cell 2.")
"""


## 2. Runtime Setup

In [ ]:
import ctypes, glob, os, sys, types
from pathlib import Path

os.environ["PYTORCH_JIT"] = "0"
os.environ["TORCHDYNAMO_DISABLE"] = "1"

import site as _site
for _pattern in [
    "/usr/local/cuda*/lib64/libnvrtc-builtins.so*",
    "/usr/local/cuda*/lib64/libnvJitLink.so*",
    "/usr/lib/x86_64-linux-gnu/libnvrtc-builtins.so*",
    "/usr/lib/x86_64-linux-gnu/libnvJitLink.so*",
    *[os.path.join(sp, "nvidia", "*", "lib", "libnvrtc-builtins.so*") for sp in _site.getsitepackages()],
    *[os.path.join(sp, "nvidia", "*", "lib", "libnvJitLink.so*")      for sp in _site.getsitepackages()],
]:
    for _lib in sorted(glob.glob(_pattern)):
        try:    ctypes.CDLL(_lib, mode=ctypes.RTLD_GLOBAL)
        except OSError: pass

import torch

if hasattr(torch._C, "_jit_set_nvfuser_enabled"):
    torch._C._jit_set_nvfuser_enabled(False)
try:
    import triton.backends as _tb
    if not hasattr(_tb, "compiler"):
        _tb.compiler = types.SimpleNamespace(AttrsDescriptor=None)
except ImportError:
    pass

import numpy as np
import transformers

try:
    transformers.BloomPreTrainedModel
except Exception:
    transformers.BloomPreTrainedModel = type("BloomPreTrainedModel", (), {})

import peft

print(f"torch       : {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"peft        : {peft.__version__}")

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device      : {DEVICE}")
if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU         : {torch.cuda.get_device_name(0)} ({props.total_memory / (1024**3):.1f} GB)")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


## 3. Configuration

**To extend training:** increase `TOTAL_EPOCHS` and re-run from here.
The notebook auto-detects the last checkpoint and continues seamlessly.
Update `MILESTONE_EPOCHS` to control which epochs get a standalone saved adapter.


In [ ]:
MODEL_NAME   = "Qwen/Qwen3.5-0.8B"
DATASET_PATH = "/kaggle/input/datasets/nachomoreratfm2/dataset-0-to-5/CloudDataset_Fixed"

# ── Epoch settings ─────────────────────────────────────────────────────────────
TOTAL_EPOCHS      = 20   # <── increase this to extend training (e.g. 25 to continue from epoch 20)
MILESTONE_EPOCHS  = [5, 10, 15, 20]  # save a standalone adapter at these epochs

# ── Directories ────────────────────────────────────────────────────────────────
OUTPUT_DIR             = "/kaggle/working/qwen35-cloud-lora"
TRAINER_CKPT_DIR       = os.path.join(OUTPUT_DIR, "trainer_checkpoints")
MILESTONE_ADAPTERS_DIR = os.path.join(OUTPUT_DIR, "milestone_adapters")

# ── Hyper-parameters ───────────────────────────────────────────────────────────
MAX_LENGTH   = 768
SEED         = 42
LEARNING_RATE = 1e-4
LORA_R       = 16
LORA_ALPHA   = 16
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

for d in [OUTPUT_DIR, TRAINER_CKPT_DIR, MILESTONE_ADAPTERS_DIR]:
    os.makedirs(d, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)

# Auto-detect dataset if the default Kaggle slug changed.
_inp = Path("/kaggle/input")
print("=== /kaggle/input datasets ===")
if _inp.exists():
    for d in sorted(_inp.iterdir()):
        print(f"  {d.name}/")
    if not os.path.isdir(DATASET_PATH):
        for p in _inp.rglob("CloudDataset_Fixed"):
            if p.is_dir():
                DATASET_PATH = str(p)
                print(f"\nAuto-detected dataset: {DATASET_PATH}")
                break

print(f"\nModel        : {MODEL_NAME}")
print(f"Dataset      : {DATASET_PATH}")
print(f"Total epochs : {TOTAL_EPOCHS}")
print(f"Milestones   : {MILESTONE_EPOCHS}")
print(f"LR           : {LEARNING_RATE:.2e}")
print(f"Trainer ckpts: {TRAINER_CKPT_DIR}")
print(f"Adapters     : {MILESTONE_ADAPTERS_DIR}")

assert os.path.isdir(DATASET_PATH), f"Dataset not found: {DATASET_PATH}"


## 4. Load and Merge Dataset Classes

In [ ]:
from datasets import ClassLabel, load_dataset

print("Loading dataset...")
dataset = load_dataset(
    "imagefolder",
    data_files={
        "train":      f"{DATASET_PATH}/train/**",
        "validation": f"{DATASET_PATH}/val/**",
        "test":       f"{DATASET_PATH}/test/**",
    },
)

original_labels = dataset["train"].features["label"].names
print(f"Original classes ({len(original_labels)}): {original_labels}")

class_mapping = {
    "Patterned Clouds":   "Altocumulus",
    "Sky (General)":      "Clear Sky",
    "Thick Dark Clouds":  "Cumulonimbus",
    "Thick White Clouds": "Cumulus",
    "Veil Clouds":        "Veil",
}

merged_labels = sorted({class_mapping.get(l, l) for l in original_labels})
label2id = {l: i for i, l in enumerate(merged_labels)}
NUM_LABELS = len(merged_labels)

def remap_example(example):
    name = original_labels[example["label"]]
    example["label"] = label2id[class_mapping.get(name, name)]
    return example

dataset = dataset.map(remap_example)
new_features = dataset["train"].features.copy()
new_features["label"] = ClassLabel(names=merged_labels)
dataset = dataset.cast(new_features)

new_labels = dataset["train"].features["label"].names
label2id  = {l: i for i, l in enumerate(new_labels)}
id2label  = {i: l for l, i in label2id.items()}

print(f"Merged classes ({NUM_LABELS}): {new_labels}")
print(f"Train: {len(dataset['train'])}, Val: {len(dataset['validation'])}, Test: {len(dataset['test'])}")


## 5. Define Classification Prompt & SFT Targets

In [ ]:
SYSTEM_PROMPT = (
    "You are a cloud classification expert with a lot of expertise on weather, physics, and clouds. "
    "Your goal is to, given a cloud image, classify it with one of the following classes:\n\n"
    + "\n".join(f"- {l}" for l in new_labels)
)
USER_PROMPT = (
    "Classify the cloud type in this image. "
    "The class must be included between the tags <answer></answer> using exactly one "
    "of the class names listed above. The tags must be placed at the end of your answer."
)
ANSWER_TEMPLATE = "<answer>{label}</answer>"

def build_messages(label_text=None):
    msgs = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user",   "content": [{"type": "image"}, {"type": "text", "text": USER_PROMPT}]},
    ]
    if label_text is not None:
        msgs.append({"role": "assistant",
                     "content": [{"type": "text", "text": ANSWER_TEMPLATE.format(label=label_text)}]})
    return msgs

print("Sample prompt built for: Cumulonimbus")
for msg in build_messages("Cumulonimbus"):
    texts = [c["text"] for c in msg["content"] if c.get("type") == "text"]
    has_img = any(c.get("type") == "image" for c in msg["content"])
    print(f"\n[{msg['role']}] {'[IMG] ' if has_img else ''}")
    for t in texts:
        print(t)


## 6. Load Model — Auto-detect Checkpoint

- **Fresh start:** no checkpoint in `trainer_checkpoints/` → creates a new LoRA adapter.
- **Resume:** checkpoint found → loads adapter weights from it; Trainer will restore
  optimizer state (Adam m/v), scheduler state (cosine position), and RNG state on `.train()`.


In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, get_peft_model, PeftModel

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_NAME)

compute_dtype = (
    torch.bfloat16 if (DEVICE == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32
)

# ── Detect whether to resume or start fresh ────────────────────────────────────
last_checkpoint = get_last_checkpoint(TRAINER_CKPT_DIR)

print(f"Loading base model ({compute_dtype})...")
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=compute_dtype,
    device_map="auto" if DEVICE == "cuda" else None,
)
if DEVICE != "cuda":
    base_model = base_model.to(DEVICE)

if last_checkpoint is not None:
    print(f"\n>>> Checkpoint found: {last_checkpoint}")
    print(">>> Loading LoRA adapter from checkpoint...")
    model = PeftModel.from_pretrained(base_model, last_checkpoint, is_trainable=True)
    print(">>> Trainer will restore optimizer state, scheduler state, and RNG on .train().")
    print("    (model weights ✓ | optimizer m/v ✓ | cosine LR position ✓ | RNG ✓)")
else:
    print("\n>>> No checkpoint found — starting fresh from epoch 0.")
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=LORA_TARGET_MODULES,
    )
    model = get_peft_model(base_model, lora_config)
    print(">>> Fresh LoRA adapter created.")

model.print_trainable_parameters()


## 7. Data Collator

In [ ]:
def apply_template(messages, add_generation_prompt):
    return processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=add_generation_prompt,
    )

class QwenVLMCollator:
    """SFT collator: masks the prompt tokens, keeps only the assistant answer tokens for loss."""
    def __init__(self, processor, max_length):
        self.processor  = processor
        self.max_length = max_length

    def __call__(self, examples):
        images, prompt_texts, full_texts = [], [], []
        for ex in examples:
            image      = ex["image"].convert("RGB")
            label_text = id2label[int(ex["label"])]
            prompt_texts.append(apply_template(build_messages(), add_generation_prompt=True))
            full_texts.append(apply_template(build_messages(label_text), add_generation_prompt=False))
            images.append(image)

        kw = dict(images=images, padding=True, truncation=True,
                  max_length=self.max_length, return_tensors="pt")
        prompt_batch = self.processor(text=prompt_texts, **kw)
        full_batch   = self.processor(text=full_texts,   **kw)

        labels = full_batch["input_ids"].clone()
        labels[full_batch["attention_mask"] == 0] = -100
        for i in range(labels.shape[0]):
            prompt_len = int(prompt_batch["attention_mask"][i].sum().item())
            labels[i, :prompt_len] = -100
        full_batch["labels"] = labels
        return full_batch

collator     = QwenVLMCollator(processor=processor, max_length=MAX_LENGTH)
sample_batch = collator([dataset["train"][0]])
print("Batch shapes:", {k: tuple(v.shape) for k, v in sample_batch.items()})
print(f"Masked tokens : {(sample_batch['labels'] == -100).sum().item()}")
print(f"Target tokens : {(sample_batch['labels'] != -100).sum().item()}")


## 8. Callbacks

### MilestoneAdapterSaver
Exports a clean, standalone LoRA adapter at the end of each milestone epoch.
These directories are **never** deleted by `save_total_limit` — they live outside
the Trainer checkpoint rotation.

### EfficiencyCallback
Records per-step timing, VRAM, and GPU power draw.


In [ ]:
import time
from transformers import TrainerCallback

# ── pynvml (power monitoring) ──────────────────────────────────────────────────
try:
    import pynvml
    pynvml.nvmlInit()
    _N_GPUS_NVML  = pynvml.nvmlDeviceGetCount()
    _nvml_handles = [pynvml.nvmlDeviceGetHandleByIndex(i) for i in range(_N_GPUS_NVML)]
    _NVML_OK = True
    print(f"pynvml ready: {_N_GPUS_NVML} GPU(s) monitored for power.")
except Exception:
    _nvml_handles = []
    _NVML_OK = False
    print("pynvml unavailable — power will be estimated from TDP.")

_N_CUDA_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0


class MilestoneAdapterSaver(TrainerCallback):
    """
    Saves the LoRA adapter to a clean standalone directory at the end of every
    milestone epoch. The directory is never touched by save_total_limit.
    """
    def __init__(self, model, milestone_epochs: list, adapter_base_dir: str):
        self.model         = model
        self.milestones    = set(milestone_epochs)
        self.adapter_base  = adapter_base_dir

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = round(state.epoch)
        if epoch in self.milestones:
            save_path = os.path.join(self.adapter_base, f"adapter_epoch{epoch:02d}")
            os.makedirs(save_path, exist_ok=True)
            self.model.save_pretrained(save_path)
            with open(os.path.join(save_path, "training_info.txt"), "w") as fh:
                fh.write(f"epoch={epoch}\n")
                fh.write(f"global_step={state.global_step}\n")
                fh.write(f"eval_loss={state.best_metric}\n")
                fh.write(f"total_epochs_planned={args.num_train_epochs}\n")
            print(f"\n[MilestoneAdapterSaver] Epoch {epoch:02d} → {save_path}")


class EfficiencyCallback(TrainerCallback):
    """Per-step timing, VRAM, and power. Summed across all GPUs."""
    def __init__(self):
        self.step_times   = []
        self.mem_alloc    = []
        self.mem_reserved = []
        self.power_w      = []
        self._t0          = None
        self._train_start = None
        self.peak_mem_gb  = 0.0

    def on_train_begin(self, args, state, control, **kwargs):
        self._train_start = time.perf_counter()
        if torch.cuda.is_available():
            for i in range(_N_CUDA_GPUS):
                torch.cuda.reset_peak_memory_stats(i)

    def on_step_begin(self, args, state, control, **kwargs):
        self._t0 = time.perf_counter()

    def on_step_end(self, args, state, control, **kwargs):
        if self._t0:
            self.step_times.append(time.perf_counter() - self._t0)
        if torch.cuda.is_available():
            alloc = [torch.cuda.memory_allocated(i) / 1e9 for i in range(_N_CUDA_GPUS)]
            rsvd  = [torch.cuda.memory_reserved(i)  / 1e9 for i in range(_N_CUDA_GPUS)]
            self.mem_alloc.append(sum(alloc))
            self.mem_reserved.append(sum(rsvd))
            peak = sum(torch.cuda.max_memory_allocated(i) for i in range(_N_CUDA_GPUS)) / 1e9
            self.peak_mem_gb = max(self.peak_mem_gb, peak)
        if _NVML_OK:
            try:
                self.power_w.append(sum(pynvml.nvmlDeviceGetPowerUsage(h) / 1000 for h in _nvml_handles))
            except Exception:
                pass

    def on_train_end(self, args, state, control, **kwargs):
        self.total_time = time.perf_counter() - self._train_start

    def summary(self):
        import numpy as _np
        print("\n" + "="*60)
        print("  Training Efficiency Summary")
        print("="*60)
        if self.step_times:
            st = _np.array(self.step_times)
            print(f"  Steps          : {len(st)}")
            print(f"  Avg step time  : {st.mean()*1000:.1f} ms")
            print(f"  Peak VRAM      : {self.peak_mem_gb:.2f} GB")
        if hasattr(self, "total_time"):
            print(f"  Total time     : {self.total_time/60:.2f} min")

milestone_cb  = MilestoneAdapterSaver(model, MILESTONE_EPOCHS, MILESTONE_ADAPTERS_DIR)
efficiency_cb = EfficiencyCallback()
print("Callbacks ready: MilestoneAdapterSaver + EfficiencyCallback")


## 9. TrainingArguments, Trainer & Training

### Key design decisions

| Setting | Value | Why |
|---------|-------|-----|
| `num_train_epochs` | `TOTAL_EPOCHS` | Configurable; set in Section 3 |
| `lr_scheduler_type` | cosine | One continuous cosine decay across all epochs |
| `warmup_ratio` | 0.1 | 10% of total steps used for warm-up |
| `save_total_limit` | 3 | Keep 3 most recent checkpoints for reliable resume |
| `load_best_model_at_end` | **False** | Milestone adapters are true epoch N weights, not best-by-loss |
| `resume_from_checkpoint` | auto-detected | Restores weights + optimizer + scheduler + RNG |

To extend training: increase `TOTAL_EPOCHS` in Section 3 and re-run from there.


In [ ]:
from transformers import Trainer, TrainingArguments

training_cuda = DEVICE == "cuda"
training_bf16 = training_cuda and torch.cuda.is_bf16_supported()
training_fp16 = training_cuda and not training_bf16

_tok_acc = {"correct": 0, "total": 0}

def compute_token_accuracy(eval_pred, compute_result=False):
    """Token accuracy on non-masked (assistant) positions."""
    preds  = eval_pred.predictions
    labels = eval_pred.label_ids
    if preds is None:
        if compute_result:
            c, t = _tok_acc["correct"], _tok_acc["total"]
            _tok_acc.update(correct=0, total=0)
            return {"accuracy": float(c / t) if t > 0 else 0.0}
        return {}
    if isinstance(preds, tuple): preds = preds[0]
    import numpy as _np
    preds  = _np.asarray(preds.detach().cpu().float() if hasattr(preds, "detach") else preds)
    labels = _np.asarray(labels.detach().cpu()         if hasattr(labels,"detach") else labels)
    if preds.ndim != 3 or preds.shape[1] < 2:
        if compute_result:
            c, t = _tok_acc["correct"], _tok_acc["total"]
            _tok_acc.update(correct=0, total=0)
            return {"accuracy": float(c / t) if t > 0 else 0.0}
        return {}
    shift_preds  = _np.argmax(preds[:, :-1, :], axis=-1)
    shift_labels = labels[:, 1:]
    mask = shift_labels != -100
    _tok_acc["correct"] += int(_np.sum((shift_preds == shift_labels) & mask))
    _tok_acc["total"]   += int(_np.sum(mask))
    if compute_result:
        c, t = _tok_acc["correct"], _tok_acc["total"]
        _tok_acc.update(correct=0, total=0)
        return {"accuracy": float(c / t) if t > 0 else 0.0}
    return {}


sft_args = TrainingArguments(
    output_dir=TRAINER_CKPT_DIR,

    # ── Epoch & batch ──────────────────────────────────────────────────────────
    num_train_epochs=TOTAL_EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,         # effective batch size = 8
    per_device_eval_batch_size=1,

    # ── Learning rate: ONE continuous cosine decay across all TOTAL_EPOCHS ──────
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,                      # warm up for 10% of ALL total steps
    weight_decay=0.01,

    # ── Logging & evaluation ───────────────────────────────────────────────────
    logging_steps=10,
    eval_strategy="epoch",

    # ── Checkpointing for session resume ──────────────────────────────────────
    # save_strategy="epoch"  →  saves optimizer.pt, scheduler.pt, rng_state.pth
    # save_total_limit=3     →  keeps the 3 most recent checkpoints
    # load_best_model_at_end=False  →  milestone adapters are TRUE epoch N weights
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=False,

    # ── Precision & misc ──────────────────────────────────────────────────────
    bf16=training_bf16,
    fp16=training_fp16,
    gradient_checkpointing=False,
    remove_unused_columns=False,
    dataloader_num_workers=2 if training_cuda else 0,
    dataloader_pin_memory=training_cuda,
    prediction_loss_only=False,
    batch_eval_metrics=True,
    report_to="none",
    seed=SEED,
)

sft_trainer = Trainer(
    model=model,
    args=sft_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=collator,
    compute_metrics=compute_token_accuracy,
    callbacks=[milestone_cb, efficiency_cb],
)

# ── CodeCarbon (optional) ──────────────────────────────────────────────────────
try:
    from codecarbon import EmissionsTracker
    _tracker = EmissionsTracker(
        project_name=f"qwen35_sft_{TOTAL_EPOCHS}epochs",
        output_dir=OUTPUT_DIR,
        log_level="error",
        save_to_file=True,
    )
    _tracker.start()
    _use_cc = True
except Exception as _e:
    print(f"CodeCarbon unavailable ({_e}), will use TDP estimate.")
    _use_cc = False

# ── Train ──────────────────────────────────────────────────────────────────────
import time as _time_mod
_t0 = _time_mod.perf_counter()

starting_epoch = 0
if last_checkpoint:
    import json as _json
    _state_path = os.path.join(last_checkpoint, "trainer_state.json")
    if os.path.exists(_state_path):
        with open(_state_path) as _f:
            starting_epoch = int(_json.load(_f).get("epoch", 0))

print(f"\nStarting training...")
print(f"  Total epochs    : {TOTAL_EPOCHS}")
print(f"  Starting epoch  : {starting_epoch}")
print(f"  Remaining epochs: {TOTAL_EPOCHS - starting_epoch}")
if last_checkpoint:
    print(f"  Resume from     : {last_checkpoint}")
    print("  Restoring       : model weights ✓ | Adam m/v ✓ | cosine LR ✓ | RNG ✓")
else:
    print("  Fresh start     : new adapter, epoch 0")

sft_result = sft_trainer.train(resume_from_checkpoint=last_checkpoint)
sft_trainer.save_state()

_elapsed = _time_mod.perf_counter() - _t0
_n_gpus  = torch.cuda.device_count() if torch.cuda.is_available() else 0

if _use_cc:
    try:
        _emissions_kg = _tracker.stop()
        _energy_kwh   = _tracker._total_energy.kWh
    except Exception:
        _use_cc = False
if not _use_cc:
    _energy_kwh   = (70 * 0.85 * max(_n_gpus, 1) * _elapsed / 3600) / 1000
    _emissions_kg = _energy_kwh * 0.233

print(f"\nTraining complete. Loss: {sft_result.training_loss:.4f}")
print(f"Total time : {_elapsed/60:.1f} min")
print(f"Energy     : {_energy_kwh:.4f} kWh  |  CO2: {_emissions_kg*1000:.2f} g CO2eq")


## 10. Inspect Saved Milestone Adapters

In [ ]:
import shutil

print("=== Milestone adapters ===")
for ep in MILESTONE_EPOCHS:
    path = os.path.join(MILESTONE_ADAPTERS_DIR, f"adapter_epoch{ep:02d}")
    if os.path.isdir(path):
        files = list(Path(path).rglob("*"))
        size_mb = sum(f.stat().st_size for f in files if f.is_file()) / 1e6
        print(f"  epoch {ep:02d}: {path}  ({size_mb:.1f} MB)")
        for f in sorted(files):
            if f.is_file():
                print(f"    {f.name}")
    else:
        print(f"  epoch {ep:02d}: NOT YET SAVED (training may not have reached this epoch)")

# Zip all milestones into /kaggle/working/ for easy download
print("\n=== Zipping milestone adapters ===")
for ep in MILESTONE_EPOCHS:
    path = os.path.join(MILESTONE_ADAPTERS_DIR, f"adapter_epoch{ep:02d}")
    if os.path.isdir(path):
        zip_path = os.path.join("/kaggle/working", f"qwen35_lora_epoch{ep:02d}_adapter")
        shutil.make_archive(zip_path, "zip", root_dir=path)
        print(f"  epoch {ep:02d}: {zip_path}.zip")


## 11. Evaluate on Test Set

In [ ]:
import re
from tqdm.auto import tqdm

GENERATION_KWARGS = {"max_new_tokens": 64, "do_sample": False}

def move_to(batch, device):
    return {k: v.to(device) if hasattr(v, "to") else v for k, v in batch.items()}

def classify_image(image):
    image = image.convert("RGB")
    prompt = apply_template(build_messages(), add_generation_prompt=True)
    inputs = processor(text=[prompt], images=[image], padding=True, return_tensors="pt")
    inputs = move_to(inputs, next(model.parameters()).device)
    with torch.no_grad():
        gen_ids = model.generate(**inputs, **GENERATION_KWARGS)
    gen_ids = gen_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(gen_ids, skip_special_tokens=True,
                                   clean_up_tokenization_spaces=False)[0].strip()

_exact = {l.lower(): l for l in new_labels}

def normalize_prediction(text):
    m = re.search(r"<answer>\s*(.*?)\s*</answer>", text, re.IGNORECASE | re.DOTALL)
    if m: text = m.group(1).strip()
    cleaned = " ".join(text.replace("\n", " ").split()).strip().lower()
    if cleaned in _exact: return _exact[cleaned]
    for l in new_labels:
        if l.lower() in cleaned or cleaned in l.lower(): return l
    toks = set(cleaned.replace("/", " ").replace("_", " ").replace("-", " ").split())
    best_l, best_s = None, 0
    for l in new_labels:
        s = len(toks & set(l.lower().replace("_", " ").replace("-", " ").split()))
        if s > best_s: best_s, best_l = s, l
    return best_l or new_labels[0]

def has_valid_answer(text):
    m = re.search(r"<answer>\s*(.*?)\s*</answer>", text, re.IGNORECASE | re.DOTALL)
    return bool(m) and " ".join(m.group(1).split()).strip().lower() in _exact

model.eval()
true_labels, pred_labels, raw_responses = [], [], []

for ex in tqdm(dataset["test"], total=len(dataset["test"])):
    raw  = classify_image(ex["image"])
    pred = normalize_prediction(raw)
    true_labels.append(id2label[int(ex["label"])])
    pred_labels.append(pred)
    raw_responses.append(raw)

accuracy_top1 = sum(t == p for t, p in zip(true_labels, pred_labels)) / len(true_labels)
format_rate   = sum(has_valid_answer(r) for r in raw_responses) / len(raw_responses)

print(f"\n{'='*50}")
print(f"  Top-1 Accuracy  : {accuracy_top1:.4f}")
print(f"  Format rate     : {format_rate:.4f}")
print(f"  Samples         : {len(true_labels)}")
print(f"{'='*50}")


## 12. Classification Report & Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(true_labels, pred_labels, labels=new_labels, digits=3, zero_division=0))

cm = confusion_matrix(true_labels, pred_labels, labels=new_labels)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=new_labels, yticklabels=new_labels, linewidths=0.5, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix — Qwen3.5-0.8B LoRA Epoch {TOTAL_EPOCHS} (Top-1: {accuracy_top1:.3f})")
plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0); plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, f"confusion_matrix_epoch{TOTAL_EPOCHS}.png")
plt.savefig(cm_path, dpi=150); plt.show()
print(f"Saved: {cm_path}")

predictions_df = pd.DataFrame({"true_label": true_labels, "predicted_label": pred_labels, "raw": raw_responses})
pred_path = os.path.join(OUTPUT_DIR, f"test_predictions_epoch{TOTAL_EPOCHS}.csv")
predictions_df.to_csv(pred_path, index=False)

per_class = (
    predictions_df.assign(correct=predictions_df["true_label"] == predictions_df["predicted_label"])
    .groupby("true_label")["correct"].mean().reindex(new_labels)
)
fig, ax = plt.subplots(figsize=(12, 6))
per_class.plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("Accuracy"); ax.set_title(f"Per-Class Accuracy — LoRA Epoch {TOTAL_EPOCHS}")
ax.axvline(1/NUM_LABELS, color="red", ls="--", label="Random baseline"); ax.legend()
ax.invert_yaxis(); plt.tight_layout()
pc_path = os.path.join(OUTPUT_DIR, f"per_class_accuracy_epoch{TOTAL_EPOCHS}.png")
plt.savefig(pc_path, dpi=150); plt.show()


## 13. Save Final Metrics & Efficiency Summary

In [ ]:
import json as _json

metrics = {
    "accuracy_top1":    accuracy_top1,
    "format_rate":      format_rate,
    "num_test_samples": len(true_labels),
    "model":            MODEL_NAME,
    "total_epochs":     TOTAL_EPOCHS,
    "milestone_epochs": MILESTONE_EPOCHS,
    "training_loss":    sft_result.training_loss,
    "energy_kwh":       _energy_kwh,
    "co2_g":            _emissions_kg * 1000,
}
m_path = os.path.join(OUTPUT_DIR, f"metrics_epoch{TOTAL_EPOCHS}.json")
with open(m_path, "w") as fh:
    _json.dump(metrics, fh, indent=2)
print(f"Saved metrics: {m_path}")

efficiency_cb.summary()

# Efficiency plots
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, (data, ylabel, title, color) in zip(axes, [
    (efficiency_cb.step_times, "Step time (ms)", "Step Time", "steelblue"),
    (efficiency_cb.mem_alloc,  "GPU Memory (GB)", "VRAM Usage", "green"),
    (efficiency_cb.power_w,    "Power (W)",       "GPU Power",  "orange"),
]):
    if data:
        arr = np.array(data) * (1000 if "ms" in ylabel else 1)
        ax.plot(arr, alpha=0.7, linewidth=0.8, color=color)
        ax.axhline(arr.mean(), color="red", ls="--", label=f"Mean: {arr.mean():.1f}")
        ax.set_xlabel("Step"); ax.set_ylabel(ylabel); ax.set_title(title); ax.legend()

plt.suptitle(f"LoRA SFT Efficiency — {TOTAL_EPOCHS} epochs", fontsize=13, fontweight="bold")
plt.tight_layout()
eff_path = os.path.join(OUTPUT_DIR, f"efficiency_plots_epoch{TOTAL_EPOCHS}.png")
plt.savefig(eff_path, dpi=150, bbox_inches="tight"); plt.show()

print(f"\n=== FINAL RESULTS ===")
print(f"Top-1 Accuracy : {accuracy_top1:.4f}")
print(f"Format rate    : {format_rate:.4f}")
print(f"Milestone adapters saved: {[os.path.join(MILESTONE_ADAPTERS_DIR, f'adapter_epoch{e:02d}') for e in MILESTONE_EPOCHS if os.path.isdir(os.path.join(MILESTONE_ADAPTERS_DIR, f'adapter_epoch{e:02d}'))]}")
